# Module 3 — Process Analytics: Severity, Reserves & Decision Support

**Business context:** Model claim amounts (severity), estimate reserves at P50/P75/P95 confidence, compute pure premium (loss cost), measure actuarial model lift (Lorenz/Gini), and audit fairness by driver age group.

---
| Component | Method | Output |
|---|---|---|
| Severity | Gamma GLM + CatBoost RMSE | MAE, RMSE |
| Frequency | Poisson GLM + CatBoost Poisson | MAE, RMSE |
| Pure premium | freq × severity | Loss cost (€/policy) |
| Reserves | CatBoost Quantile P50/P75/P95 | Reserve bands |
| Actuarial lift | Lorenz curve, Gini coefficient | Portfolio ranking |
| Geographic risk | Regional aggregation | Risk map input |
| Fairness | Disparate impact by age group | EU-directive audit |

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import mlflow
import mlflow.catboost
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Project data layer
from claims.data import load_fremtpl2, build_claims_dataset, claims_only
from claims.features import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    catboost_cat_indices,
)
from claims.evaluation import (
    regression_report_dict,
    gini_coefficient,
    plot_lorenz_curve,
)

# Process analytics modules
from claims.process.severity import (
    gamma_glm_pipeline,
    catboost_severity,
    poisson_glm_pipeline,
    catboost_frequency,
)
from claims.process.reserving import (
    fit_reserve_models,
    reserve_predictions,
    plot_reserve_bands,
    RESERVE_QUANTILES,
)
from claims.process.decisions import (
    pure_premium,
    pure_premium_summary,
    regional_risk_profile,
    fairness_audit,
    plot_fairness_bars,
)

print("All imports successful.")
print(f"Reserve quantiles: {RESERVE_QUANTILES}")

## 2. Data Preparation

In [ ]:
print("Loading freMTPL2 from OpenML (cached after first download)...")
freq, sev = load_fremtpl2()
print(f"Frequency table: {freq.shape}  |  Severity table: {sev.shape}")

# Build combined dataset with engineered targets
df_all = build_claims_dataset(freq, sev)
print(f"Combined dataset: {df_all.shape}")

# Severity analysis requires policies with at least one claim
df_sev = claims_only(df_all).copy()
df_sev = df_sev[df_sev["AvgSeverity"].notna() & (df_sev["AvgSeverity"] > 0)]
print(f"Severity subset (claims with positive severity): {df_sev.shape}")

# Severity distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_sev["AvgSeverity"], bins=100, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Average Severity (€)")
axes[0].set_ylabel("Count")
axes[0].set_title("Severity Distribution (linear scale)")

axes[1].hist(np.log1p(df_sev["AvgSeverity"]), bins=100, color="darkorange", edgecolor="white")
axes[1].set_xlabel("log1p(Average Severity)")
axes[1].set_ylabel("Count")
axes[1].set_title("Severity Distribution (log scale — closer to normal)")

plt.tight_layout()
plt.show()

print(f"\nSeverity summary:")
print(df_sev["AvgSeverity"].describe().round(2))

## 3. Severity Modeling

### 3a. Train / Test Split for Severity

In [ ]:
# Target: AvgSeverity (mean claim amount per policy)
y_sev = df_sev["AvgSeverity"].values

# Split
train_idx, test_idx = train_test_split(
    np.arange(len(df_sev)), test_size=0.2, random_state=42
)

df_sev_train = df_sev.iloc[train_idx].reset_index(drop=True)
df_sev_test  = df_sev.iloc[test_idx].reset_index(drop=True)
y_sev_train  = y_sev[train_idx]
y_sev_test   = y_sev[test_idx]

print(f"Severity train: {df_sev_train.shape}  |  test: {df_sev_test.shape}")

### 3b. Gamma GLM (Actuarial Baseline)

In [ ]:
print("Fitting Gamma GLM pipeline (TweedieRegressor power=2, log link)...")
glm_sev_pipe = gamma_glm_pipeline()
glm_sev_pipe.fit(df_sev_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES], y_sev_train)

glm_sev_pred = glm_sev_pipe.predict(df_sev_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
glm_sev_pred = np.clip(glm_sev_pred, 0, None)  # severity must be non-negative

glm_sev_metrics = regression_report_dict(y_sev_test, glm_sev_pred)
print("\nGamma GLM — Severity Metrics:")
for k, v in glm_sev_metrics.items():
    print(f"  {k:<6}: {v:,.2f}")

### 3c. CatBoost Severity

In [ ]:
print("Fitting CatBoost RMSE severity model (raw string categoricals)...")

# CatBoost receives a DataFrame with raw string categoricals
X_sev_train = df_sev_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_sev_test  = df_sev_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

# Ensure numerics are float
for col in NUMERIC_FEATURES:
    X_sev_train[col] = pd.to_numeric(X_sev_train[col], errors="coerce")
    X_sev_test[col]  = pd.to_numeric(X_sev_test[col], errors="coerce")

# cat_features as list of column name strings
cb_sev_model = catboost_severity(iterations=500, learning_rate=0.05, depth=6, verbose=0)
cb_sev_model.fit(
    X_sev_train, y_sev_train,
    eval_set=(X_sev_test, y_sev_test),
    verbose=100,
)

cb_sev_pred = cb_sev_model.predict(X_sev_test)
cb_sev_pred = np.clip(cb_sev_pred, 0, None)

cb_sev_metrics = regression_report_dict(y_sev_test, cb_sev_pred)
print("\nCatBoost RMSE — Severity Metrics:")
for k, v in cb_sev_metrics.items():
    print(f"  {k:<6}: {v:,.2f}")

### 3d. Severity Model Comparison

In [ ]:
sev_compare = pd.DataFrame({
    "Model": ["Gamma GLM", "CatBoost RMSE"],
    "MAE":  [glm_sev_metrics["mae"], cb_sev_metrics["mae"]],
    "RMSE": [glm_sev_metrics["rmse"], cb_sev_metrics["rmse"]],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(sev_compare["Model"], sev_compare["MAE"], color=["steelblue", "darkorange"])
axes[0].set_title("Severity: MAE Comparison")
axes[0].set_ylabel("MAE (€)")
for i, v in enumerate(sev_compare["MAE"]):
    axes[0].text(i, v + 10, f"{v:,.0f}", ha="center", fontsize=10)

axes[1].bar(sev_compare["Model"], sev_compare["RMSE"], color=["steelblue", "darkorange"])
axes[1].set_title("Severity: RMSE Comparison")
axes[1].set_ylabel("RMSE (€)")
for i, v in enumerate(sev_compare["RMSE"]):
    axes[1].text(i, v + 10, f"{v:,.0f}", ha="center", fontsize=10)

plt.suptitle("Severity Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(sev_compare.to_string(index=False))

## 4. Frequency Modeling

### 4a. Train / Test Split for Frequency

In [ ]:
# Frequency uses the full dataset (all policies, including zero-claim)
y_freq = df_all["ClaimFrequency"].values

train_idx_f, test_idx_f = train_test_split(
    np.arange(len(df_all)), test_size=0.2, random_state=42
)
df_freq_train = df_all.iloc[train_idx_f].reset_index(drop=True)
df_freq_test  = df_all.iloc[test_idx_f].reset_index(drop=True)
y_freq_train  = y_freq[train_idx_f]
y_freq_test   = y_freq[test_idx_f]

print(f"Frequency train: {df_freq_train.shape}  |  test: {df_freq_test.shape}")
print(f"Mean claim frequency: {y_freq.mean():.4f} claims/year")

### 4b. Poisson GLM

In [ ]:
print("Fitting Poisson GLM pipeline (TweedieRegressor power=1, log link)...")
glm_freq_pipe = poisson_glm_pipeline()
glm_freq_pipe.fit(df_freq_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES], y_freq_train)

glm_freq_pred = glm_freq_pipe.predict(df_freq_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
glm_freq_pred = np.clip(glm_freq_pred, 0, None)

glm_freq_metrics = regression_report_dict(y_freq_test, glm_freq_pred)
print("\nPoisson GLM — Frequency Metrics:")
for k, v in glm_freq_metrics.items():
    print(f"  {k:<6}: {v:.6f}")

### 4c. CatBoost Poisson

In [ ]:
print("Fitting CatBoost Poisson frequency model (raw string categoricals)...")

X_freq_train = df_freq_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_freq_test  = df_freq_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

for col in NUMERIC_FEATURES:
    X_freq_train[col] = pd.to_numeric(X_freq_train[col], errors="coerce")
    X_freq_test[col]  = pd.to_numeric(X_freq_test[col], errors="coerce")

cb_freq_model = catboost_frequency(iterations=500, learning_rate=0.05, depth=6, verbose=0)
cb_freq_model.fit(
    X_freq_train, y_freq_train,
    eval_set=(X_freq_test, y_freq_test),
    verbose=100,
)

cb_freq_pred = cb_freq_model.predict(X_freq_test)
cb_freq_pred = np.clip(cb_freq_pred, 0, None)

cb_freq_metrics = regression_report_dict(y_freq_test, cb_freq_pred)
print("\nCatBoost Poisson — Frequency Metrics:")
for k, v in cb_freq_metrics.items():
    print(f"  {k:<6}: {v:.6f}")

### 4d. Frequency Model Comparison

In [ ]:
freq_compare = pd.DataFrame({
    "Model": ["Poisson GLM", "CatBoost Poisson"],
    "MAE":  [glm_freq_metrics["mae"], cb_freq_metrics["mae"]],
    "RMSE": [glm_freq_metrics["rmse"], cb_freq_metrics["rmse"]],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(freq_compare["Model"], freq_compare["MAE"], color=["steelblue", "darkorange"])
axes[0].set_title("Frequency: MAE Comparison")
axes[0].set_ylabel("MAE (claims/year)")
for i, v in enumerate(freq_compare["MAE"]):
    axes[0].text(i, v + 0.001, f"{v:.5f}", ha="center", fontsize=10)

axes[1].bar(freq_compare["Model"], freq_compare["RMSE"], color=["steelblue", "darkorange"])
axes[1].set_title("Frequency: RMSE Comparison")
axes[1].set_ylabel("RMSE (claims/year)")
for i, v in enumerate(freq_compare["RMSE"]):
    axes[1].text(i, v + 0.001, f"{v:.5f}", ha="center", fontsize=10)

plt.suptitle("Frequency Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(freq_compare.to_string(index=False))

## 5. Pure Premium (Loss Cost)

In [ ]:
# For the pure premium we need overlapping rows between frequency and severity test sets.
# We use the full-dataset frequency model predictions and pair with severity predictions
# on all policies (severity = 0 for non-claimants).

# Predict frequency on full dataset
X_all = df_all[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
for col in NUMERIC_FEATURES:
    X_all[col] = pd.to_numeric(X_all[col], errors="coerce")

freq_pred_all = np.clip(cb_freq_model.predict(X_all), 0, None)

# Predict severity on full dataset (use Gamma GLM for out-of-sample stability)
sev_pred_all  = np.clip(
    glm_sev_pipe.predict(df_all[NUMERIC_FEATURES + CATEGORICAL_FEATURES]), 0, None
)

# Pure premium = freq x severity
pp_pred_all = pure_premium(freq_pred_all, sev_pred_all)

df_all["PredFrequency"] = freq_pred_all
df_all["PredSeverity"]  = sev_pred_all
df_all["PredPurePremium"] = pp_pred_all

print("Pure Premium Summary (portfolio-level):")
exposure_all = pd.to_numeric(df_all["Exposure"], errors="coerce").fillna(0).values
pp_summary = pure_premium_summary(
    df_all["PurePremium"].values,
    pp_pred_all,
    exposure_all,
)
print(pp_summary.to_string(index=False))

In [ ]:
# Scatter: actual PurePremium vs predicted (sample 2000 points for readability)
sample_idx = np.random.default_rng(0).integers(0, len(df_all), size=min(2000, len(df_all)))
pp_actual_sample = df_all["PurePremium"].values[sample_idx]
pp_pred_sample   = pp_pred_all[sample_idx]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pp_pred_sample, pp_actual_sample, alpha=0.3, s=10, color="steelblue")
max_val = max(pp_pred_sample.max(), pp_actual_sample.max())
ax.plot([0, max_val], [0, max_val], "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Predicted Pure Premium (€)")
ax.set_ylabel("Actual Pure Premium (€)")
ax.set_title("Actual vs Predicted Pure Premium (random sample n=2,000)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Reserve Estimation with Uncertainty

In [ ]:
# Reserve models are trained on the claims-only severity subset (quantile regression)
print(f"Reserve quantiles: {RESERVE_QUANTILES}")
print("Fitting quantile reserve models...")

# Use the numeric+categorical columns that catboost_cat_indices() references
X_res_train = df_sev_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_res_test  = df_sev_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

for col in NUMERIC_FEATURES:
    X_res_train[col] = pd.to_numeric(X_res_train[col], errors="coerce")
    X_res_test[col]  = pd.to_numeric(X_res_test[col], errors="coerce")

# fit_reserve_models returns a dict {quantile: CatBoostRegressor}
reserve_models = fit_reserve_models(
    X_res_train.values, y_sev_train,
    X_res_test.values, y_sev_test,
    quantiles=RESERVE_QUANTILES,
)
print(f"Reserve models fitted: {list(reserve_models.keys())}")

In [ ]:
# Generate reserve predictions on test set
reserves_df = reserve_predictions(X_res_test.values, reserve_models)

print("Reserve predictions sample (first 10 rows):")
print(reserves_df.head(10).round(2).to_string(index=False))
print(f"\nPortfolio reserve summary:")
print(reserves_df.describe().round(2))

In [ ]:
# Reserve bands visualisation
fig, ax = plt.subplots(figsize=(10, 6))
plot_reserve_bands(y_sev_test, reserves_df, n_samples=300, ax=ax)
plt.tight_layout()
plt.show()

# Coverage check: what % of actuals fall below each quantile?
for col, q in zip(["P50", "P75", "P95"], RESERVE_QUANTILES):
    coverage = (y_sev_test <= reserves_df[col].values).mean()
    print(f"  {col} coverage: {coverage:.1%}  (target: {q:.0%})")

## 7. Actuarial Lift — Lorenz Curve + Gini

In [ ]:
# Lorenz curve compares two model rankings on the frequency test set:
#   - Poisson GLM (actuarial baseline)
#   - CatBoost Poisson (ML challenger)
y_freq_test_binary = (y_freq_test > 0).astype(int)  # binarise for Gini / Lorenz

glm_gini = gini_coefficient(y_freq_test_binary, glm_freq_pred)
cb_gini  = gini_coefficient(y_freq_test_binary, cb_freq_pred)

print(f"Gini coefficient — Poisson GLM   : {glm_gini:.4f}")
print(f"Gini coefficient — CatBoost Poisson: {cb_gini:.4f}")
print(f"CatBoost lift over GLM: {(cb_gini - glm_gini) / abs(glm_gini) * 100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 6))
plot_lorenz_curve(
    y_freq_test_binary, glm_freq_pred, model_name="Poisson GLM", ax=ax
)
plot_lorenz_curve(
    y_freq_test_binary, cb_freq_pred,  model_name="CatBoost Poisson", ax=ax
)
plt.tight_layout()
plt.show()

## 8. Geographic Risk Heatmap

In [ ]:
# Aggregate predicted risk metrics by Region
risk_profile = regional_risk_profile(
    df_all,
    region_col="Region",
    freq_pred_col="PredFrequency",
    sev_pred_col="PredSeverity",
)

print(f"Regions in dataset: {risk_profile.shape[0]}")
print("\nTop 5 highest-risk regions by predicted pure premium:")
print(risk_profile.head(5).round(4).to_string(index=False))

# Bar chart of predicted pure premium — top 15 regions
top15 = risk_profile.head(15)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(
    top15["Region"].astype(str),
    top15["pred_pure_premium"],
    color=plt.cm.Reds(np.linspace(0.4, 0.9, len(top15))),
)
ax.set_xlabel("Region")
ax.set_ylabel("Predicted Pure Premium (€/policy)")
ax.set_title("Top 15 Regions by Predicted Pure Premium")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

> **Note:** In production, the `regional_risk_profile()` output would be joined to a French region GeoJSON and rendered as a choropleth using Plotly:
> ```python
> import plotly.express as px
> fig = px.choropleth(
>     risk_profile,
>     geojson=french_regions_geojson,
>     locations="Region",
>     color="pred_pure_premium",
>     color_continuous_scale="Reds",
>     title="Predicted Pure Premium by French Region",
> )
> fig.show()
> ```

## 9. Fairness Audit

In [ ]:
# Create driver age buckets: <25, 25-40, 40-60, 60+
df_all["DrivAge_num"] = pd.to_numeric(df_all["DrivAge"], errors="coerce")
df_all["AgeBucket"] = pd.cut(
    df_all["DrivAge_num"],
    bins=[0, 25, 40, 60, 200],
    labels=["<25", "25-40", "40-60", "60+"],
    right=False,
)

bucket_counts = df_all["AgeBucket"].value_counts().sort_index()
print("Driver age bucket distribution:")
for bucket, cnt in bucket_counts.items():
    print(f"  {bucket:<10}: {cnt:,} policies ({cnt / len(df_all):.1%})")

In [ ]:
# Run fairness audit: compare predicted frequency vs actual claim rate by age group
audit_df = fairness_audit(
    df_all,
    pred_col="PredFrequency",
    group_col="AgeBucket",
    actual_col="HasClaim",
)

print("Fairness Audit by Driver Age Group:")
print(audit_df.to_string(index=False))

# Plot
fig = plot_fairness_bars(audit_df, group_col="AgeBucket", figsize=(10, 4))
plt.show()

### Fairness Interpretation — EU Directive Context

The `disparate_impact_ratio` column measures each age group's predicted rate relative to the portfolio average:

- A ratio of **1.0** indicates no disparity from the portfolio mean.
- Ratios **> 1.25** or **< 0.80** are conventionally considered evidence of potential disparate impact ("80% rule", widely applied in US EEOC and increasingly referenced in EU fairness guidelines).

**EU Directive 2004/113/EC** prohibits the use of gender as a rating factor in insurance pricing (confirmed by the ECJ *Test-Achats* ruling, 2011). More broadly, Solvency II and the forthcoming **EU AI Act (Article 10)** require insurers to:

1. Document training data characteristics that may introduce proxy discrimination.
2. Monitor model outputs for disparate impact on protected characteristics.
3. Maintain explainability sufficient for supervisory audit (EIOPA guidelines).

**Driver age** is a permitted rating factor under current EU law (unlike gender), but models should be audited to ensure that age-correlated proxies (e.g., `VehBrand`, `Area`) do not inadvertently encode impermissible characteristics. Age groups showing disparate impact ratios outside the 0.80–1.25 corridor warrant further actuarial review before deployment.

## 10. MLflow Logging

In [ ]:
mlflow.set_experiment("process-analytics")

with mlflow.start_run(run_name="severity-frequency-reserve"):

    # --- Severity metrics ---
    mlflow.log_params({
        "severity_glm": "TweedieRegressor(power=2,link=log)",
        "severity_cb_iterations": 500,
        "severity_cb_depth": 6,
        "frequency_glm": "TweedieRegressor(power=1,link=log)",
        "frequency_cb_iterations": 500,
        "frequency_cb_depth": 6,
        "reserve_quantiles": str(RESERVE_QUANTILES),
        "reserve_cb_iterations": 400,
    })

    mlflow.log_metrics({
        # Severity
        "sev_glm_mae":  glm_sev_metrics["mae"],
        "sev_glm_rmse": glm_sev_metrics["rmse"],
        "sev_cb_mae":   cb_sev_metrics["mae"],
        "sev_cb_rmse":  cb_sev_metrics["rmse"],
        # Frequency
        "freq_glm_mae":  glm_freq_metrics["mae"],
        "freq_glm_rmse": glm_freq_metrics["rmse"],
        "freq_cb_mae":   cb_freq_metrics["mae"],
        "freq_cb_rmse":  cb_freq_metrics["rmse"],
        # Actuarial lift
        "gini_glm": glm_gini,
        "gini_cb":  cb_gini,
        "gini_lift_pct": (cb_gini - glm_gini) / abs(glm_gini) * 100,
    })

    # Log the CatBoost severity and frequency models
    mlflow.catboost.log_model(cb_sev_model,  artifact_path="catboost_severity")
    mlflow.catboost.log_model(cb_freq_model, artifact_path="catboost_frequency")

    # Log the GLM pipelines
    mlflow.sklearn.log_model(glm_sev_pipe,  artifact_path="glm_severity")
    mlflow.sklearn.log_model(glm_freq_pipe, artifact_path="glm_frequency")

    run_id = mlflow.active_run().info.run_id

print(f"MLflow run logged: {run_id}")
print("Experiment: process-analytics")
print(f"  sev_cb_mae   : {cb_sev_metrics['mae']:,.2f}")
print(f"  freq_cb_mae  : {cb_freq_metrics['mae']:.6f}")
print(f"  gini_cb      : {cb_gini:.4f}")
print(f"  gini_glm     : {glm_gini:.4f}")